# Chat loop and token budget

Why does the API forget what you asked one call earlier? And what does it cost to send all IKEA reviews to a model? Guess first, then measure.

In this notebook, we will:
1. Ask a follow-up question and see that the API forgets.
2. Keep a list of messages and send it again with every question.
3. Send the same prompt three times and compare the answers.
4. Count tokens with `tiktoken` and compute the cost of reading all IKEA reviews.
5. Compare our token count with the usage the API reports, with low and high reasoning effort.

**Time:** 10 minutes, together. The notebook makes 10 small API calls.

**Self-check:** allow about 5 additional minutes for the questions at the end of the core exercise. Optional extensions take extra time.

## Libraries and settings

`ask()` sends one request to the model. It works like `client.responses.create()` of the OpenAI library, with two extras:

- It removes personal details from the text before sending (see notebook 01).
- It saves every answer in the folder `.cache`. Repeating the same completed request with the same `run` uses its saved answer. Changing the prompt, settings or run number can send a new request.

In [ ]:
# Libraries
import os
import platform
from datetime import datetime
from platform import python_version

import pandas as pd
import tiktoken

# Course helpers: DATA is the folder 'data', ask() sends a request to the model
from llm_course import DATA, ask

# Model and course price assumption in USD per 1 million input tokens
MODEL = 'gpt-6-luna'
PRICE_INPUT_PER_M = 0.10

# Check that the API key is available (never print the key itself)
print('API key found:', bool(os.environ.get('OPENAI_API_KEY')))

# Show current working directory
print(os.getcwd())

## Step 1: Ask the API a follow-up question

We send two separate requests. Does the model understand what "it" means in the second question?

In [ ]:
# First question
r1 = ask(
    model=MODEL,
    input='Name one typical dish in the IKEA restaurant. Answer in one sentence.',
)
print('Answer 1:', r1.output_text)

# Follow-up question, sent as a separate request
r2 = ask(model=MODEL, input='Why is it so popular?')
print('Answer 2:', r2.output_text)

The second request contains no earlier question. The model may guess a plausible meaning of "it", but it has not seen the previous request. **These requests are independent: our code has not supplied any history.**

## Step 2: Keep a list of messages and send it again

A chat application like ChatGPT keeps the whole conversation in a list and sends the full list with every new question. Each message has a `role` (`system`, `user` or `assistant`) and a `content`.

In [ ]:
# The system message holds the rules for the whole conversation
RULES = 'You answer questions about IKEA. Keep every answer to one sentence.'
messages = [{'role': 'system', 'content': RULES}]

# The questions of our conversation
conversation = [
    'Name one typical dish in the IKEA restaurant.',
    'Why is it so popular?',
    'What was my first question?',
]

for question in conversation:
    # 1. Add the question to the list
    messages.append({'role': 'user', 'content': question})
    # 2. Send the whole list
    r = ask(model=MODEL, input=messages)
    # 3. Add the answer to the list
    messages.append({'role': 'assistant', 'content': r.output_text})

    print('Q:', question)
    print('A:', r.output_text)
    print('   input tokens of this request:', r.usage.input_tokens, '\n')

Now the model can answer the follow-up questions, because the earlier questions and answers are part of the input. The price: the input grows with every turn, so every turn costs more input tokens.

## Step 3: Same prompt, different answers

The model picks the next token with weighted dice. So the same prompt can give different answers. Different values of `run` request separate answers. Repeating an existing run reuses its saved answer.

In [ ]:
# The same prompt, three times
for i in [1, 2, 3]:
    r = ask(model=MODEL, input='Describe IKEA in exactly five words.', run=i)
    print('Run', i, ':', r.output_text)

## Step 4: What does it cost to read all IKEA reviews?

We want to send every review to the model with this prompt: *"Positive, negative or mixed? Answer with one word."*

Guess the cost **before** you compute anything.

In [ ]:
# Your guess in USD (before you compute anything!)
my_guess_usd = None  # TODO: e.g. 0.50

### What is a token?

A model does not read letters or words, but **tokens**: whole words, parts of words or punctuation marks. The price of a request depends on the number of tokens. `tiktoken` is the tokenizer of the OpenAI models.

In [ ]:
# Load the tokenizer
enc = tiktoken.get_encoding('o200k_base')

# A review from IKEA Philadelphia (1 of 5 stars, original spelling)
review = 'Swedish meatballs good. Attitude at.Return/Exchange counter bad.'

# Text -> token IDs (numbers)
token_ids = enc.encode(review)
print('Words: ', len(review.split()))
print('Tokens:', len(token_ids))
print('IDs:   ', token_ids)

# Token IDs -> pieces of text, one token at a time
pieces = [enc.decode([token_id]) for token_id in token_ids]
print('Pieces:', pieces)

**Your turn:** try a German sentence, a very long word and a text with typos. What happens to the number of tokens?

In [ ]:
# TODO: replace the text and run the cell again
my_text = 'Wir alle lieben IKEA, solange man von niemandem Hilfe braucht.'

token_ids = enc.encode(my_text)
print('Words: ', len(my_text.split()))
print('Tokens:', len(token_ids))
print('Pieces:', [enc.decode([token_id]) for token_id in token_ids])

### Count the tokens of all IKEA reviews

Counting happens on this computer. No review is sent to the API in this step, so we can use the original file.

In [ ]:
# Read all IKEA reviews
df = pd.read_csv(DATA / 'ikea_reviews.csv')

# Number of tokens per review
df['tokens'] = [len(enc.encode(text)) for text in df['text']]

print(df['tokens'].describe().round(1))
print('Total tokens:', f"{df['tokens'].sum():,}")

### Compute the cost and compare it with your guess

In [ ]:
# The prompt is sent together with every review
PROMPT = 'Positive, negative or mixed? Answer with one word.\n'
prompt_tokens = len(enc.encode(PROMPT))

# Input tokens of all requests: all reviews + one prompt per review
input_tokens = df['tokens'].sum() + len(df) * prompt_tokens

# Cost: tokens / 1 million x price per 1 million tokens
cost = input_tokens / 1_000_000 * PRICE_INPUT_PER_M

print(f'Input tokens:  {input_tokens:,}')
print(f'Cost (input):  {cost:.4f} USD')
print(f'Your guess:    {my_guess_usd} USD')

The tokens of the answers (output tokens) come on top; they are more expensive than input tokens.

### Context window

The context window is the maximum number of tokens in one request: input plus output. For this exercise, use a context budget of 1,000,000 tokens. Reserve space for the prompt, message formatting and output as well as the review texts.

## Step 5: Usage and reasoning effort

Send the same short review with low and high reasoning effort. Compare the label, input tokens and output tokens with the local count. The usage also reports how many output tokens were used for hidden reasoning.

In [ ]:
print('Local input-token count:', len(enc.encode(PROMPT + review)))

for effort in ['low', 'high']:
    r = ask(model=MODEL, input=PROMPT + review, reasoning={'effort': effort})
    print('Effort:', effort, '| answer:', r.output_text)
    print('Input:', r.usage.input_tokens, '| output:', r.usage.output_tokens,
          '| of these, reasoning:', r.usage.output_tokens_details.reasoning_tokens)

The API count includes message formatting. Its output-token count can include hidden reasoning tokens even when the visible answer is one word. Higher reasoning effort can increase output tokens and cost; a useful quality improvement must be measured.

## Check your understanding

Take about 5 minutes. Answer in one or two sentences per question, using the outputs above where useful. No additional model calls are needed. Try first, then open the hints.

1. Why can the model refer to an earlier question in step 2? Does that mean its weights changed during the conversation?
2. Use one tokenization output to explain why counting words is not enough to estimate the input cost.
3. Why must the cost calculation count the prompt once per review? Which costs are still missing from the input-only estimate?
4. Why can repeated requests give different answers? Why would one run be weak evidence when comparing prompts?
5. How can a one-word answer use more than one output token? How could you decide whether higher reasoning effort is worth its cost?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. The application resends the message history. The model sees it in the current input; its weights do not change.

2. Tokens can be words, word pieces or punctuation. Their number depends on the text and tokenizer; refer to your measured pieces.

3. Each separate request contains its own prompt. Output tokens, including hidden reasoning tokens, cost extra; message formatting also adds input tokens.

4. Sampling can choose different next tokens. Compare several runs on the same inputs with the same model and settings.

5. The usage can include hidden reasoning tokens and answer punctuation. Compare measured correctness, output tokens and response time on several examples.

</details>

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')